1. **Ты обучил пять разных моделей на абсолютно одних и тех же обучающих данных, и все они дают precision 95%. Есть ли шанс, что объединив эти модели, можно получить лучший результат? Если да — как? Если нет — почему?**

2. **В чём разница между классификаторами с жёстким (hard voting) и мягким (soft voting) голосованием?**

3. **Можно ли ускорить обучение bagging-ансамбля, распределив его по нескольким серверам? А pasting-ансамбля, boosting-ансамбля, случайного леса (random forest) или stacking-ансамбля?**

4. **В чём польза оценки out-of-bag (OOB)?**

5. **Что делает ансамбли extra-trees более случайными, чем обычные случайные леса? Как эта дополнительная случайность помогает? Классификаторы extra-trees медленнее или быстрее обычных случайных лесов?**

6. **Если твой AdaBoost-ансамбль недообучается (underfits) на обучающих данных, какие гиперпараметры стоит покрутить и в какую сторону?**

7. **Если твой gradient boosting ансамбль переобучается (overfits) на обучающей выборке, learning rate нужно увеличить или уменьшить?**

8. **Загрузи датасет MNIST (из главы 3) и раздели его на обучающую, валидационную и тестовую выборки (например, 50 000 экземпляров на обучение, 10 000 на валидацию и 10 000 на тест). Затем обучи разные классификаторы: случайный лес, extra-trees и SVM. Дальше попробуй объединить их в ансамбль, который на валидационной выборке превосходит каждый отдельный классификатор, используя soft или hard voting. Когда такой ансамбль найдётся — проверь его на тестовой выборке. Насколько он лучше каждого отдельного классификатора?**

9. **Прогони отдельные классификаторы из предыдущего упражнения, чтобы получить предсказания на валидационной выборке, и создай из этих предсказаний новую обучающую выборку: каждый экземпляр — это вектор из предсказаний всех твоих классификаторов для одного изображения, а целевое значение — класс этого изображения. Обучи классификатор на этой новой выборке. Поздравляем — ты обучил блендер (blender), и вместе с классификаторами он образует stacking-ансамбль! Теперь оцени ансамбль на тестовой выборке. Для каждого изображения из теста сделай предсказания всеми классификаторами, затем подай эти предсказания в блендер, чтобы получить предсказание ансамбля. Как это сравнить с voting-классификатором, который ты обучил раньше? Теперь попробуй то же самое с `StackingClassifier`. Получается лучше? Если да — почему?**

In [ ]:
# MNIST dataset
# 50k train
# 10k valid
# 10k testing
# Train a Random Forest Classifier, extra-trees, SVM.

# Combine ensemble that outperforms each individual classifier on the valid set
# using a soft/hard voting
# When find a best try perform on test set
# How much better does it perform compared to the individual classifiers?

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import LinearSVC
import numpy as np

mnist = fetch_openml("mnist_784", as_frame=False)
X, y = mnist.data, mnist.target
# Training and test set (it's already is splitted and shuffled)
X_train, y_train = X[:50000], y[:50000]
X_valid, y_valid = X[50000:60000], y[50000:60000]
X_test, y_test = X[60000:], y[60000:]
print(X_train.shape, X_valid.shape, X_test.shape)

# Ниже я запускал каждый отдельно для измерения без Voting

# Random Forest 0.9744
rnd_clf = RandomForestClassifier(n_estimators=500,
                                 n_jobs=-1, random_state=42)
# rnd_clf.fit(X_train, y_train)
# y_pred_rf = rnd_clf.predict(X_valid)
# print(rnd_clf.score(X_valid, y_valid))

# Extra trees 0.976
et_clf = ExtraTreesClassifier(n_estimators=500,
                                 n_jobs=-1, random_state=42)
# et_clf.fit(X_train, y_train)
# y_pred_et = et_clf.predict(X_valid)
# print(et_clf.score(X_valid, y_valid))

# LinearSVC без dual 0.0991, с dual 0.8662
lsvc_clf = LinearSVC(random_state=42, max_iter=1000, tol=20, dual=True)
# lsvc_clf.fit(X_train, y_train)
# y_pred_lsvc = lsvc_clf.predict(X_valid)
# print(lsvc_clf.score(X_valid, y_valid))

(50000, 784) (10000, 784) (10000, 784)


In [ ]:
# Проверь у себя: напечатай y_valid.dtype и y_valid[:5]. 
# И скажи до запуска: что даст np.array(['10', '3']).astype(np.int64) — ошибку или [10 3]?
voting_clf = VotingClassifier(
    estimators=[
        ("rf", rnd_clf),
        ("et", et_clf),
    ]
)
voting_clf.fit(X_train, y_train)

# Ниже будет объяснение почему это важно
rf = voting_clf.named_estimators_["rf"]
print("rf predict без np.int:",rf.predict(X_valid[:5]))
print("оригинальный y valid: ",y_valid[:5])


# Вывод score каждого классификатора
for name, clf in voting_clf.named_estimators_.items():
    # Если передать y_valid как есть, то print начнет давать 0.0
    # Какая-то тема что строки сравниваются с цифрой, хотя voting самому это не мешает почему-то (мб сам меняет)
    # но в случае когда руками каждый classifier отдельно, то нужно делать преорабзование из-за нашего датасета
    y_valid_enc = y_valid.astype(np.int64)
    print(name, "=", clf.score(X_valid, y_valid_enc))

print(voting_clf.score(X_valid, y_valid))

rf predict без np.int: [3 8 6 9 6]
оригинальный y valid:  ['3' '8' '6' '9' '6']
rf = 0.9744
et = 0.976
0.9755
